# US equities panel: refitting the selected configuration for the holdout

Every number this case study has produced so far was measured on the validation folds, and
every choice was made by looking at them: which model family, which configuration, how many
names to hold on each side of the book, how to size them, which risk control to overlay, what
to charge for a trade. A result selected that way cannot also be the evidence that the
selection was sound, because the ranking and the evidence would be the same measurement.

The holdout is the window nothing has been selected on: `evaluation.holdout_start` through
`evaluation.holdout_end` in `config/setup.yaml`, which no notebook from
[`16_backtest`](16_backtest.ipynb) through [`19_costs`](19_costs.ipynb) reads. This notebook
fits the selected configuration again on the history available before that window opens and
writes its predictions over it. It publishes predictions and nothing else:
[`21_holdout_backtest`](21_holdout_backtest.ipynb) turns them into a return series, and
[`22_strategy_analysis`](22_strategy_analysis.ipynb) reads both back with intervals.

**A holdout prediction is not the validation model scored on a later window.** Section 2 fits
again, over a training interval that ends before the window opens, and the new training
identity is what makes the refit visible rather than asserted: the identity covers the CV
interval, so a run that came back with the validation training hash would mean no refit
happened. Section 3 checks exactly that, and raises.

**Learning objectives**


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 美國股票面板：為保留樣本重新擬合選定設定

此案例研究至今產生的每個數值都在驗證折上量測，所有選擇也都依據驗證折：選擇模型類別、設定、部位組合每邊持有多少檔股票、如何決定部位大小、套用哪項風險控制，以及每筆交易要計入多少成本。依此方式選出的結果，不能再當成選擇正確的證據，因為排序與證據會是同一次量測。

保留樣本是未用於任何選擇的窗口：從 `evaluation.holdout_start` 到 `evaluation.holdout_end` 的期間，定義於 `config/setup.yaml`；從 [`16_backtest`](16_backtest.ipynb) 到 [`19_costs`](19_costs.ipynb) 都未讀取。本 Notebook 在窗口開始前可用的歷史資料上重新擬合選定設定，並寫出窗口內的預測。它只發布預測：[`21_holdout_backtest`](21_holdout_backtest.ipynb) 將其轉為報酬序列，[`22_strategy_analysis`](22_strategy_analysis.ipynb) 則連同區間讀回兩者。

**保留樣本預測不是把驗證模型拿到較晚窗口評分。** 第 2 節會在窗口開始前結束的訓練區間重新擬合；新的訓練識別資訊使重新擬合可被查證，而不只是宣稱。識別資訊涵蓋交叉驗證區間，因此若執行回傳驗證訓練雜湊值，就表示沒有重新擬合。第 3 節會明確檢查，遇到此情況便拋出錯誤。

**學習目標**
<!-- ML4T_ZH_END c0000-p000 -->

- Derive a holdout retraining interval from declarations rather than choosing one, and say
  which declaration supplies each boundary.
- Explain why the interval's upper bound is a label horizon below the window's start, and what
  a zero gap would leak.
- Read a holdout prediction set as a measurement of one configuration rather than a comparison
  among several.
- Say why a notebook that evaluates a second configuration on this window refuses instead of
  replacing the first.

**Book reference**: Chapter 20, Section 20.2

**Prerequisites**: [`19_costs`](19_costs.ipynb) is the last stage that selects; the
configuration it prices is the one refitted here.

**What it writes**: one training run and one prediction set, both at `split='holdout'`. No
backtest, no selection, no comparison.

<!-- ML4T_ZH_BEGIN c0000-p001 -->
- 從宣告推導保留樣本重新訓練區間，而非任意選擇，並指出各邊界來自哪項宣告。
- 解釋區間上界為何須比窗口起點早一個標籤預測期間，以及零間隔會洩漏什麼。
- 將保留樣本預測集合解讀為單一設定的量測，而非多項設定的比較。
- 說明為何在此窗口評估第二項設定時，Notebook 會拒絕而非取代第一項。

**書籍參考**：第 20 章第 20.2 節。

**先決條件**：[`19_costs`](19_costs.ipynb) 是最後一個進行選擇的階段；它重新計價的設定，就是此處重新擬合的設定。

**產出內容**：在 `split='holdout'` 各寫出一次訓練執行與一組預測。沒有回測、選擇或比較。
<!-- ML4T_ZH_END c0000-p001 -->


In [1]:
"""US equities panel: refit the selected configuration and predict the holdout window."""

import polars as pl

from case_studies.research import open_study
from case_studies.research.holdout import build_holdout_training_spec
from case_studies.research.models import reconstruct_locked_model_request
from case_studies.utils.registry import training_hash_from_spec
from case_studies.utils.strategy_analysis import (
    holdout_generations_to_retire,
    registered_holdout_generations,
    resolve_solvent_carrier,
)
from utils.paths import get_case_study_dir

In [2]:
CASE_STUDY_ID = "us_equities_panel"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""

In [3]:
study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)

## 1. Which configuration the holdout runs

The holdout runs the configuration this case study reports, resolved through the same
`resolve_solvent_carrier` that [`19_costs`](19_costs.ipynb) prices. Resolving it again here
rather than passing a hash along is deliberate: the two notebooks then agree by construction,
and a hash written down in one and read in the other agrees only until the sweep is rebuilt.

Nothing about the holdout enters this choice. The selected configuration is the cross-stage
validation rank-1 over the baseline, allocation and risk-overlay stages, and it was fixed
before this notebook ran. Which stage it comes from is printed rather than assumed, because on
this panel the leader is not always the risk-overlay run.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 1. 保留樣本使用哪項設定

保留樣本使用本案例研究報告的設定，透過 `resolve_solvent_carrier` 解析；[`19_costs`](19_costs.ipynb) 重新計價時也使用同一個解析方法。此處再次解析，而非傳遞雜湊值，是刻意設計：如此兩份 Notebook 依建構方式保持一致；若在一處寫下雜湊值、另一處讀取，只有在掃描尚未重建時才一致。

保留樣本的任何資訊都不參與這項選擇。選定設定是在基準、配置與風險覆蓋三個階段間，依驗證結果排名第一者，並在本 Notebook 執行前固定。此處會印出其來源階段，而非預設一定來自風險覆蓋，因為在此面板中，領先者不一定是覆蓋後的執行。
<!-- ML4T_ZH_END c0004-p000 -->


In [4]:
carrier = resolve_solvent_carrier(CASE_STUDY_ID)
print(
    f"Selected configuration: {carrier['val_backtest_hash']}  stage={carrier['val_stage']}  "
    f"family={carrier['family']}  config={carrier['config_name']}  "
    f"label={carrier['label']}"
)
print(
    f"  validation Sharpe {carrier['val_sharpe']:.3f}, max drawdown {carrier['max_drawdown']:.3f}"
)
print(f"  fitted by training run {carrier['training_hash']}")

Selected configuration: 3076efd14f7c  stage=signal  family=gbm  config=leaves_63_mse  label=fwd_ret_1d
  validation Sharpe 1.053, max drawdown -0.990
  fitted by training run 5c884d63d89c


The checkpoint is part of the configuration. Where a family publishes a prediction set per
checkpoint on a declared schedule, the selected configuration's prediction set names one of
them, and refitting without it would produce the model at the end of training rather than the
one that was ranked. A family with no checkpoint dimension stores NULL in both columns, and
that NULL is carried through unchanged.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
檢查點是設定的一部分。若某模型類別依宣告時程，為每個檢查點發布一組預測，選定設定的預測集合便指定其中一個；重新擬合時若不保留它，就會得到訓練結束時的模型，而非原先參與排名者。沒有檢查點維度的模型類別，在兩個欄位都儲存 NULL，並原樣傳遞。
<!-- ML4T_ZH_END c0006-p000 -->


In [5]:
validation_prediction = study.results.open(carrier["val_prediction_hash"])
prediction_record = validation_prediction.registry_record()
CHECKPOINT_KIND = prediction_record["checkpoint_kind"]
CHECKPOINT_VALUE = prediction_record["checkpoint_value"]
print(f"Checkpoint: {CHECKPOINT_KIND}={CHECKPOINT_VALUE}")

Checkpoint: iteration=500


## 2. The window, and the model that is allowed to see it

The holdout window is not a choice made here. It is `evaluation.holdout_start` and
`evaluation.holdout_end` from this case study's own `setup.yaml`, read through the same
`canonical_window` the fold derivation and the backtest slice both go through, so the three
cannot disagree.

The training interval is everything available before that window, bounded above by a label
buffer. **The size of that buffer depends on which label the selection landed on.** This panel
declares three - `fwd_ret_1d`, `fwd_ret_5d` and `fwd_ret_21d` - with buffers of one, five and
twenty-one sessions, and the derivation reads the buffer of the selected label rather than
defaulting one. A row dated `t` records an outcome that is not known until `t` plus the
horizon, so a training set running to the day the window opens would be fitted on labels whose
returns resolve inside it. Both boundaries are printed below so the gap can be read rather
than taken on the derivation's word.

Everything else about the configuration is carried across unchanged, and the fields that
cannot be - the eligibility manifest, and any parameter this family resolves from a fold's own
training rows - are recomputed against the holdout fold. Carrying those forward would fit a
model keyed to the validation folds and call it a retrain.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 2. 窗口及允許查看窗口的模型

保留樣本窗口不是在此處選擇。它由 `evaluation.holdout_start` 與 `evaluation.holdout_end` 界定，兩者來自本案例研究的 `setup.yaml`，並透過推導驗證折與切分回測資料所用的同一個 `canonical_window` 讀取，因此三處不會產生分歧。

訓練區間涵蓋該窗口之前所有可用資料，並在上界保留標籤緩衝期。**緩衝期長度取決於選到哪個標籤。** 此面板宣告三種標籤：`fwd_ret_1d`、`fwd_ret_5d` 與 `fwd_ret_21d`，分別緩衝一、五與二十一個交易時段；推導時讀取選定標籤的緩衝期，不套用統一預設值。日期為 `t` 的資料列，其結果要到 `t` 加上預測期間後才會確定；因此若訓練集延伸到窗口開始當天，就會用到報酬在窗口內才確定的標籤。下方印出兩側邊界，讓讀者直接檢查間隔，而非只相信推導結果。

設定的其他內容均原樣保留；無法沿用的欄位，包括合格性清單，以及此模型類別根據各折訓練資料列解析的參數，都會針對保留樣本折重新計算。若直接沿用，就會擬合以驗證折為鍵的模型，卻稱之為重新訓練。
<!-- ML4T_ZH_END c0008-p000 -->


In [6]:
observation_timeline = (
    pl.read_parquet(study.root / "labels" / f"{carrier['label']}.parquet")
    .get_column("timestamp")
    .unique()
    .sort()
    .to_list()
)
validation_spec = study.results.open(carrier["training_hash"]).spec()
holdout_spec = build_holdout_training_spec(
    study,
    validation_spec,
    timeline=observation_timeline,
    case_study=CASE_STUDY_ID,
)

fold = holdout_spec["computation"]["cv"]["folds"][0]
print(f"Holdout fold {fold['fold']}")
print(f"  trains   {fold['train_start']} -> {fold['train_end']}")
print(f"  predicts {fold['val_start']} -> {fold['val_end']}")
print(f"  label buffer: {holdout_spec['computation']['cv']['request']['label_buffer']}")

# The validation folds are what the buffer is measured against, and the last of them ends before
# the holdout opens. Printing both is what lets a reader check the gap.
validation_folds = validation_spec["computation"]["cv"]["folds"]
latest_validation_end = max(str(entry["val_end"]) for entry in validation_folds)
print(f"Validation folds: {len(validation_folds)}, latest evaluation end {latest_validation_end}")
print(f"Holdout training ends {fold['train_end']}, holdout opens {fold['val_start']}")

Holdout fold 16
  trains   1990-01-30 -> 2015-12-01
  predicts 2016-01-01 -> 2018-03-31
  label buffer: 21D
Validation folds: 16, latest evaluation end 2015-12-30 00:00:00
Holdout training ends 2015-12-01, holdout opens 2016-01-01


## 3. Fit, and register the predictions

`reconstruct_locked_model_request` builds the request from the specification above. Its name
comes from a locked holdout path this case study does not use; it takes a training
specification and a checkpoint, not a lock, and it is used here because it is the one call
that refuses a request that is not exactly the specification it was handed - the training
identity, the checkpoint schedule, the feature lineage and the runtime parameters are all
checked before anything is fitted.

The training identity below is new. It has to be: it covers the CV interval, and the holdout
fold is not one of the sixteen validation folds. A run that came back with the validation
training hash would mean the refit did not happen, so that is checked rather than assumed.

**The window carries one configuration, and this notebook has no way past that.** The check
below is on the selected configuration rather than on the notebook, and it has exactly two
outcomes. With the selected configuration unchanged this is an idempotent replay: the
derivation is deterministic and the training identity covers it, so the same identity comes
back and the fit is served from the registry, which is why re-running is free. With the
selected configuration changed it refuses, names both configurations, and stops.


<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 3. 擬合並登錄預測

`reconstruct_locked_model_request` 根據上述規格建立請求。它的名稱來自本案例研究未使用的鎖定保留樣本路徑；該呼叫接收訓練規格與檢查點，而非鎖定資訊。此處使用它，是因為只有這個呼叫會拒絕與所給規格不完全一致的請求：訓練識別資訊、檢查點時程、特徵來源資訊及執行環境參數，都會在擬合前檢查。

下方的訓練識別資訊是新的，必須如此：它涵蓋交叉驗證區間，而保留樣本折不屬於十六個驗證折。若執行回傳驗證訓練雜湊值，就表示沒有重新擬合，因此會明確檢查，而非直接假設。

**此窗口只允許一項設定，本 Notebook 無法繞過。** 下方檢查針對選定設定，而非只針對 Notebook，且只有兩種結果。選定設定不變時，這是可重複的重播：推導具確定性，訓練識別資訊涵蓋推導結果，因此會回傳相同識別資訊，擬合結果由登錄系統提供，重新執行幾乎沒有成本。若選定設定改變，檢查會拒絕、指出兩項設定，然後停止。
<!-- ML4T_ZH_END c0010-p000 -->

It refuses rather than offering a replacement switch, and the reason is that a replacement
would not be one. Deleting the earlier generation's rows does not undo having observed its
result: the selection that produced the new configuration may have been informed by the old
holdout number, and no deletion reaches that. A switch here would let the case study take a
second look at the window while leaving a registry that shows only one, which is the specific
thing that would make the out-of-sample claim false rather than merely weak.

<!-- ML4T_ZH_BEGIN c0010-p001 -->
它選擇拒絕，而非提供取代開關，原因是取代並不能抹去先前結果。刪除上一代資料列，無法撤銷已觀察到的結果：產生新設定的選擇，可能已受到舊保留樣本數值影響，而刪除無法消除這點。若此處提供開關，案例研究就能對窗口看第二次，登錄系統卻只顯示一次；這會使樣本外主張失真，而不只是證據變弱。
<!-- ML4T_ZH_END c0010-p001 -->


In [7]:
holdout_training_hash = training_hash_from_spec(holdout_spec)
this_generation = (holdout_training_hash, (CHECKPOINT_KIND, CHECKPOINT_VALUE))
retire = holdout_generations_to_retire(CASE_DIR, this_generation=this_generation)
# A row whose training run records no CV split cannot be shown either way, and deleting on that
# would discard a result nothing has established is wrong. It stops the run instead.
if retire.unattributable:
    raise RuntimeError(
        "the holdout window carries prediction sets whose training runs record no CV split, "
        "so whether they were refitted for the holdout cannot be established: "
        + ", ".join(
            f"{row['prediction_hash']} (training {row['training_hash']})"
            for row in retire.unattributable
        )
        + ". Establish what produced them before registering another evaluation on the same "
        "window; this notebook will not delete a row it cannot show is not a holdout result."
    )
# A row whose training run declares a non-holdout CV may not be reported as a holdout result,
# and it is also not something to delete unattended: `generate_holdout` refits on a holdout fold
# and then registers the predictions under the VALIDATION training identity, so this record
# covers both a validation-fitted model published over the window and a real refit filed under
# the wrong identity.
if retire.not_out_of_sample:
    raise RuntimeError(
        "the holdout window carries prediction sets whose training runs declare a CV split "
        "other than the holdout: "
        + ", ".join(
            f"{row['prediction_hash']} ({row['config_name']}, training {row['training_hash']})"
            for row in retire.not_out_of_sample
        )
        + ". Each is either a validation-fitted model published over the window, which is not "
        "an out-of-sample result, or a refit registered under its validation training identity, "
        "which the retired `20_strategy_synthesis/holdout.py::generate_holdout` wrote until it "
        "was deleted on 2026-09-12 - and the registry cannot tell those apart. This notebook has no way past that: establish which it is and "
        "resolve it through the registry's own lifecycle, which records that the row was retired."
    )
superseded = list(retire.superseded)
if superseded:
    raise RuntimeError(
        "the holdout window already carries a refit of a different configuration: "
        + ", ".join(
            f"{row['prediction_hash']} ({row['config_name']}, training {row['training_hash']})"
            for row in superseded
        )
        + f". This run would evaluate {carrier['config_name']} (training "
        f"{holdout_training_hash}, checkpoint {CHECKPOINT_KIND}={CHECKPOINT_VALUE}) on the same "
        "window, which would be a second configuration measured on a period this case study "
        "reports as unseen. This notebook has no way past that: deleting the earlier generation "
        "would not undo having observed it, and the selection bias it introduces is not removed "
        "by removing the rows. Either leave the selection where it was, or retire the earlier "
        "evaluation through the registry's own lifecycle, which records that a second look was "
        "taken."
    )

In [8]:
request = reconstruct_locked_model_request(
    study,
    holdout_spec,
    checkpoint_kind=CHECKPOINT_KIND,
    checkpoint_value=CHECKPOINT_VALUE,
)
model_run = request.run()
holdout_prediction = model_run.predictions[0]

if model_run.training.hash == carrier["training_hash"]:
    raise RuntimeError(
        "the holdout refit produced the validation training identity "
        f"{carrier['training_hash']}, which means it did not refit"
    )
print(f"Holdout training run:   {model_run.training.hash}")
print(f"Holdout prediction set: {holdout_prediction.hash}")

      fold 16: training n_train=8,716,098 n_val=1,208,615 trees=500 num_leaves=63 obj=regression


      fold 16: done in 140s


Holdout training run:   0a4c916e2937
Holdout prediction set: 192a4ddac3c1


What the prediction set covers, read back from the registry rather than from the request. The
two agree only if the fit published what it declared, and the counts are what a reader checks
the window against: this is a daily panel, so the session count is trading days in the window
and the row count is those sessions times the names eligible on each.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
此處從登錄系統讀回預測集合的涵蓋範圍，而非直接使用請求內容。只有擬合確實發布所宣告的內容時，兩者才會一致；讀者可用計數核對窗口：這是每日面板，因此交易時段數是窗口內的交易日數，資料列數則為各時段符合資格股票的數量加總。
<!-- ML4T_ZH_END c0013-p000 -->


In [9]:
record = holdout_prediction.registry_record()
predictions = holdout_prediction.load()
print(
    f"split={record['split']}  checkpoint={record['checkpoint_kind']}={record['checkpoint_value']}"
)
print(f"rows={predictions.height:,}  sessions={predictions['timestamp'].n_unique():,}")
print(
    f"  {predictions['timestamp'].min()} -> {predictions['timestamp'].max()}, "
    f"{predictions['symbol'].n_unique():,} names"
)

split=holdout  checkpoint=iteration=500
rows=1,208,615  sessions=560
  2016-01-04 00:00:00+00:00 -> 2018-03-26 00:00:00+00:00, 2,674 names


Every holdout prediction set the registry holds, and whether the model behind it was refitted
for the window. All of them are listed rather than one silently preferred, because the
registry is immutable and a reader looking at it later will see whatever is there. A row
marked VALIDATION-FITTED is not an out-of-sample result whatever its numbers say.

<!-- ML4T_ZH_BEGIN c0015-p000 -->
列出登錄系統中的每組保留樣本預測，及背後模型是否為此窗口重新擬合。此處全部列出，而非默默偏好其中一組，因為登錄系統不可變，日後讀者會看到其中所有內容。標為 VALIDATION-FITTED 的資料列，不論數值如何，都不是樣本外結果。
<!-- ML4T_ZH_END c0015-p000 -->


In [10]:
for row in registered_holdout_generations(CASE_DIR):
    note = (
        "refitted for the holdout" if row["refitted"] else "VALIDATION-FITTED - not out of sample"
    )
    print(
        f"  {row['prediction_hash']}  training={row['training_hash']}  {row['config_name']}  {note}"
    )

  192a4ddac3c1  training=0a4c916e2937  leaves_63_mse  refitted for the holdout


## What this notebook establishes, and what it does not

It establishes one thing: a prediction set over the holdout window, produced by the
configuration this case study selected, fitted on data that ends a full label horizon before
the window opens. That is a precondition for an out-of-sample claim, not the claim itself.
Nothing here says whether the predictions are any good - they have not been scored, sized or
traded.

It does not make the holdout a fresh test in the strict sense. The configuration reached this
notebook through a selection made on the validation folds, and this window is being used once
per configuration that gets here. What it does remove is the specific circularity of scoring a
validation-fitted model on the period meant to judge it.

Re-running this notebook is free: the same configuration re-derives the same training identity
and the fit is served from the registry. Evaluating a different configuration is not, and is
refused above.

**Next:** [`21_holdout_backtest`](21_holdout_backtest.ipynb).

<!-- ML4T_ZH_BEGIN c0017-p000 -->
## 本 Notebook 證明什麼，以及未證明什麼

它證明一件事：此案例研究選定的設定，在保留樣本窗口開始前至少一個完整標籤預測期間就結束的資料上擬合，並產生涵蓋該窗口的預測集合。這是樣本外主張的先決條件，而非主張本身。此處不說明預測是否準確；它們尚未評分、配置部位或進行交易。

它也不表示保留樣本在嚴格意義下是全新的測試。設定是經驗證折上的選擇進入本 Notebook，而每個進入此處的設定都會使用這個窗口一次。此流程消除的是特定循環問題：拿在驗證資料上擬合的模型，對原本要評判它的期間評分。

重新執行本 Notebook 幾乎沒有成本：相同設定會重新推導相同訓練識別資訊，擬合結果由登錄系統提供。評估不同設定則不行，上方會拒絕。

**下一步：** [`21_holdout_backtest`](21_holdout_backtest.ipynb)。
<!-- ML4T_ZH_END c0017-p000 -->
